In [ ]:
!pip install uv

In [ ]:
!uv pip install fastapi uvicorn pyngrok nest-asyncio evaluate rouge_score peft transformers torchao==0.16.0

In [ ]:
import os
import re
import torch
import evaluate
import nest_asyncio
import uvicorn
import asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import (
    PegasusForConditionalGeneration, PegasusTokenizer,
    BartForConditionalGeneration, BartTokenizer,
    AutoModelForSequenceClassification, AutoTokenizer
)
from peft import PeftModel

print("Loading models and tokenizers...")
device = "cuda" if torch.cuda.is_available() else "cpu"
rouge = evaluate.load("rouge")

pgs_tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
tokenizer_finbert = AutoTokenizer.from_pretrained("ProsusAI/finbert")

pgs_base = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
bart_base = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
finbert_base = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert", num_labels=3)

pgs_model_tuned = PeftModel.from_pretrained(pgs_base, "dankekw/pegasus-xsum-finance").to(device)
bart_model_tuned = PeftModel.from_pretrained(bart_base, 'dankekw/bart-finance').to(device)
finbert_model_tuned = PeftModel.from_pretrained(finbert_base, 'dankekw/finbert-tuned2').to(device)

pgs_model_tuned.eval()
bart_model_tuned.eval()
finbert_model_tuned.eval()

print(f"Все модели успешно загружены на {device.upper()}!")

SUMM_MODELS = {
    "pegasus": {"model": pgs_model_tuned, "tokenizer": pgs_tokenizer},
    "bart": {"model": bart_model_tuned, "tokenizer": bart_tokenizer}
}

def process_summarization(text, target_summary=None, min_length=50, max_length=150):
    output = {}

    clean_text = re.sub(r'[\r\n]+', ' ', text)

    for model_name, components in SUMM_MODELS.items():
        model = components["model"]
        tokenizer = components["tokenizer"]

        inputs = tokenizer(clean_text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                min_length=min_length,
                max_length=max_length,
                num_beams=5,
                no_repeat_ngram_size=3
            )

        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        result = {"summary": summary}

        if target_summary:
            scores = rouge.compute(predictions=[summary], references=[target_summary])
            result["rouge"] = {k: float(v) for k, v in scores.items()}
        else:
            result["rouge"] = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

        output[model_name] = result

    return output

def process_sentiment(text):
    id2label = {0: "negative", 1: "neutral", 2: "positive"}

    inputs = tokenizer_finbert(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = finbert_model_tuned(**inputs)

    preds = torch.argmax(outputs.logits, dim=1).item()
    return id2label[preds]

app = FastAPI(title="Financial News Analysis API")

class AnalysisRequest(BaseModel):
    article_text: str
    target_summary: str = None

@app.post("/analyze")
def analyze_article(payload: AnalysisRequest):
    summaries = process_summarization(
        text=payload.article_text,
        target_summary=payload.target_summary
    )

    bart_summary_text = summaries['bart']['summary']
    sentiment = process_sentiment(bart_summary_text)

    return {
        "summaries": summaries,
        "sentiment": sentiment
    }

NGROK_TOKEN = "your_token"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000)
print("\n" + "="*50)
print(f"server run successfully")
print(f"connection url: {public_url.public_url}")
print("="*50 + "\n")

nest_asyncio.apply()
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

In [ ]:
!fuser -k 8000/tcp